In [0]:
%sql
DESCRIBE SCHEMA EXTENDED dbjahan.bronze_data;

database_description_item,database_description_value
Catalog Name,dbjahan
Namespace Name,bronze_data
Comment,Connection to bronze data tier
Location,abfss://bronze@datatiersjahan.dfs.core.windows.net/__unitystorage/schemas/d16122e1-4944-4fc4-869e-b7ee13aa89ce
Owner,jahan.razzaghrostami@yh.nackademin.se
RootLocation,abfss://bronze@datatiersjahan.dfs.core.windows.net/
Properties,"((unity.catalog.managed.delta.defaults.defaultClusterByAuto,true), (unity.catalog.managed.delta.defaults.delta.checkpointPolicy,v2), (unity.catalog.managed.delta.defaults.delta.enableRowTracking,true), (unity.catalog.managed.delta.defaults.delta.feature.catalogManaged,supported), (unity.catalog.managed.delta.defaults.delta.parquet.format.version,2.12.0), (unity.catalog.managed.delta.defaults.delta.parquet.format.version.afe.internal,2.12.0), (unity.catalog.managed.iceberg.defaults.delta.feature.catalogManaged,supported))"
Predictive Optimization,ENABLE (inherited from METASTORE metastore_azure_swedencentral)


In [0]:
%sql
USE dbjahan.bronze_data;

CREATE OR REPLACE TABLE dbjahan.bronze_data.nyreg_bilar_raw (
  region          STRING,    
  region_text     STRING,   
  drivmedel       STRING,   
  drivmedel_text  STRING,   
  tid             STRING,   
  antal           BIGINT,   
  hamtad_tid      TIMESTAMP
)
USING DELTA;

In [0]:
import requests
import itertools
import time
from datetime import datetime
from pyspark.sql import Row

def hamta(api_namn, url, params=None, timeout=90):
    senaste = None
    for forsok in range(1, 4):
        try:
            svar = requests.get(url, params=params, timeout=timeout)
            if svar.status_code == 200:
                return svar
            senaste = str(svar.status_code)
        except requests.RequestException as fel:
            senaste = str(fel)
        if forsok < 3:
            time.sleep(5)
    raise RuntimeError(
        f"{api_namn} svarade {senaste} efter 3 försök. URL: {url}"
    )

url = "https://statistikdatabasen.scb.se/api/v2/tables/TAB3277/data"
params = {
    "lang": "sv",
    "outputFormat": "json-stat2",
    "valueCodes[ContentsCode]": "TK1001AA",
    "valueCodes[Region]": "*",
    "valueCodes[Drivmedel]": "*",
    "valueCodes[Tid]": "from(2022M01)",
}

svar = hamta("SCB TAB3277", url, params=params, timeout=90)
js = svar.json()

ids = js["id"]
dims = []
for var in ids:
    cat = js["dimension"][var]["category"]
    index = cat["index"]
    labels = cat.get("label", {})
    codes = [None] * len(index)
    for kod, plats in index.items():
        codes[plats] = (kod, labels.get(kod, kod))
    dims.append(codes)

nu = datetime.now()
rader = []
for i, combo in enumerate(itertools.product(*dims)):
    d = {var: kod for var, (kod, _txt) in zip(ids, combo)}
    t = {var: txt for var, (_kod, txt) in zip(ids, combo)}
    val = js["value"][i]
    rader.append(Row(
        region=d["Region"],
        region_text=t["Region"],
        drivmedel=d["Drivmedel"],
        drivmedel_text=t["Drivmedel"],
        tid=d["Tid"],
        antal=int(val) if val is not None else None,
        hamtad_tid=nu,
    ))

df = spark.createDataFrame(rader)
df.createOrReplaceTempView("nyreg_nya")

spark.sql("""
INSERT INTO dbjahan.bronze_data.nyreg_bilar_raw
SELECT n.region, n.region_text, n.drivmedel, n.drivmedel_text,
       n.tid, n.antal, n.hamtad_tid
FROM nyreg_nya n
WHERE NOT EXISTS (
  SELECT 1
  FROM dbjahan.bronze_data.nyreg_bilar_raw t
  WHERE t.region    = n.region
    AND t.drivmedel = n.drivmedel
    AND t.tid       = n.tid
)
""")

print("Klart")

Klart


In [0]:
%sql
SELECT * FROM dbjahan.bronze_data.nyreg_bilar_raw
LIMIT 20;

region,region_text,drivmedel,drivmedel_text,tid,antal,hamtad_tid
00,Riket,100,bensin,2022M01,5104,2026-09-27T16:07:12.176Z
00,Riket,100,bensin,2022M02,5368,2026-09-27T16:07:12.176Z
00,Riket,100,bensin,2022M03,6737,2026-09-27T16:07:12.176Z
00,Riket,100,bensin,2022M04,5968,2026-09-27T16:07:12.176Z
00,Riket,100,bensin,2022M05,7786,2026-09-27T16:07:12.176Z
00,Riket,100,bensin,2022M06,6490,2026-09-27T16:07:12.176Z
00,Riket,100,bensin,2022M07,5672,2026-09-27T16:07:12.176Z
00,Riket,100,bensin,2022M08,5975,2026-09-27T16:07:12.176Z
00,Riket,100,bensin,2022M09,5016,2026-09-27T16:07:12.176Z
00,Riket,100,bensin,2022M10,4523,2026-09-27T16:07:12.176Z


In [0]:
%sql
SELECT *
FROM dbjahan.bronze_data.nyreg_bilar_raw
WHERE region = '0180'          -- Stockholm
  AND drivmedel = '120'        -- el, om 120 är el i den här tabellen
ORDER BY tid
LIMIT 30;

region,region_text,drivmedel,drivmedel_text,tid,antal,hamtad_tid
0180,Stockholm,120,el,2022M01,709,2026-09-27T16:07:12.176Z
0180,Stockholm,120,el,2022M02,779,2026-09-27T16:07:12.176Z
0180,Stockholm,120,el,2022M03,1560,2026-09-27T16:07:12.176Z
0180,Stockholm,120,el,2022M04,910,2026-09-27T16:07:12.176Z
0180,Stockholm,120,el,2022M05,1011,2026-09-27T16:07:12.176Z
0180,Stockholm,120,el,2022M06,1327,2026-09-27T16:07:12.176Z
0180,Stockholm,120,el,2022M07,610,2026-09-27T16:07:12.176Z
0180,Stockholm,120,el,2022M08,730,2026-09-27T16:07:12.176Z
0180,Stockholm,120,el,2022M09,1219,2026-09-27T16:07:12.176Z
0180,Stockholm,120,el,2022M10,1008,2026-09-27T16:07:12.176Z


In [0]:
%sql
DESCRIBE TABLE EXTENDED dbjahan.bronze_data.nyreg_bilar_raw;

col_name,data_type,comment
region,string,null
region_text,string,null
drivmedel,string,null
drivmedel_text,string,null
tid,string,null
antal,bigint,null
hamtad_tid,timestamp,null
,,
# Delta Statistics Columns,,
Column Names,"region_text, hamtad_tid, antal, region, tid, drivmedel_text, drivmedel",
